# 05 — Structured Outputs Across Providers — Practical Examples

**Covers**: OpenAI, Anthropic, Gemini JSON modes, `instructor` library, LiteLLM unified API, side-by-side comparison

In [ ]:
import os, json, re
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Optional
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. Shared Schema — Same Pydantic Model for All Providers

In [ ]:
# Define ONCE, use across all providers
class ProductReview(BaseModel):
    product_name: str
    rating: int = Field(ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"]
    pros: list[str]
    cons: list[str]
    recommend: bool

REVIEW_TEXT = """
I bought the Sony WH-1000XM5 headphones last month. The noise cancellation is absolutely 
amazing - I can't hear anything in my loud office. Sound quality is superb, very detailed 
bass and clear highs. Battery lasts 30+ hours. The only drawbacks are the price ($350 is steep)
and they're a bit tight after 3 hours. I'd still rate them 4/5 and definitely recommend them.
"""

print('Shared schema defined: ProductReview')

## 2. OpenAI — Native Strict Structured Output

In [ ]:
# OpenAI: native .parse() with Pydantic
def extract_openai(text: str) -> ProductReview:
    r = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a product review analyzer."},
            {"role": "user", "content": text}
        ],
        response_format=ProductReview
    )
    return r.choices[0].message.parsed

openai_result = extract_openai(REVIEW_TEXT)
print(f"OpenAI Result:")
print(f"  Product:   {openai_result.product_name}")
print(f"  Rating:    {openai_result.rating}/5")
print(f"  Sentiment: {openai_result.sentiment}")
print(f"  Pros:      {openai_result.pros}")
print(f"  Cons:      {openai_result.cons}")
print(f"  Recommend: {openai_result.recommend}")

## 3. Anthropic Claude — Prompt-Based JSON + Manual Parse

In [ ]:
# Simulate the Anthropic approach with prompt engineering
# (Works equally well on the real Anthropic API)

def extract_anthropic_style(text: str) -> dict:
    """
    Simulate Anthropic's prompt-based approach.
    Real code would use: anthropic.Anthropic().messages.create(...)
    """
    # Build explicit JSON schema description for Claude
    schema_desc = json.dumps(ProductReview.model_json_schema(), indent=2)
    
    # Use OpenAI with a strict system message (simulating Claude's approach)
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": f"""You are a product review analyzer.
Return ONLY valid JSON matching this schema EXACTLY. No prose, no markdown fences.
Schema:
{schema_desc}"""
            },
            {"role": "user", "content": text}
        ]
    )
    raw = r.choices[0].message.content.strip()
    # Strip markdown fences (Claude sometimes adds these)
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()
    return json.loads(cleaned)

claude_result = extract_anthropic_style(REVIEW_TEXT)
print(f"Anthropic-style Result:")
print(f"  Product:   {claude_result.get('product_name')}")
print(f"  Rating:    {claude_result.get('rating')}/5")
print(f"  Sentiment: {claude_result.get('sentiment')}")
print(f"  Recommend: {claude_result.get('recommend')}")

## 4. Gemini — JSON Mode via `response_mime_type`

In [ ]:
# Gemini JSON mode simulation
# Real Gemini code:
# import google.generativeai as genai
# model = genai.GenerativeModel('gemini-1.5-flash',
#     generation_config={'response_mime_type': 'application/json'})
# response = model.generate_content(f'Extract review: {text}')
# data = json.loads(response.text)

# Simulate Gemini's best-effort schema via OpenAI JSON mode
def extract_gemini_style(text: str) -> dict:
    r = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},   # Gemini JSON mode equivalent
        messages=[
            {
                "role": "system",
                "content": 'Extract product review info as JSON. Fields: product_name(str), rating(1-5 int), sentiment(positive/negative/neutral), pros(list[str]), cons(list[str]), recommend(bool)'
            },
            {"role": "user", "content": text}
        ]
    )
    return json.loads(r.choices[0].message.content)

gemini_result = extract_gemini_style(REVIEW_TEXT)
print(f"Gemini-style Result:")
print(f"  Product:   {gemini_result.get('product_name')}")
print(f"  Rating:    {gemini_result.get('rating')}/5")
print(f"  Type check: rating is {type(gemini_result.get('rating')).__name__}  (not guaranteed!)")

## 5. `instructor` Library — Unified Pydantic Across All Providers

In [ ]:
# instructor wraps any provider with Pydantic support
# pip install instructor

try:
    import instructor
    
    # Patch OpenAI client with instructor
    instructor_client = instructor.from_openai(OpenAI())
    
    result = instructor_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You analyze product reviews."},
            {"role": "user", "content": REVIEW_TEXT}
        ],
        response_model=ProductReview   # instructor equivalent of response_format
    )
    
    print(f"instructor Result (OpenAI):")
    print(f"  Product:   {result.product_name}")
    print(f"  Rating:    {result.rating}/5")
    print(f"  Sentiment: {result.sentiment}")
    print(f"  type: {type(result).__name__}  ← Python object directly!")
    
except ImportError:
    print("📌 Install instructor: pip install instructor")
    print("   Then: instructor_client = instructor.from_openai(OpenAI())")
    print("         result = instructor_client.chat.completions.create(response_model=ProductReview)")

## 6. Side-by-Side Comparison — Results from All Providers

In [ ]:
import time

def benchmark_extraction(text: str):
    """Compare all approaches on the same text."""
    results = {}
    
    # OpenAI strict
    t = time.perf_counter()
    r = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": text}],
        response_format=ProductReview
    )
    results["OpenAI Strict"] = {
        "latency_ms": round((time.perf_counter() - t) * 1000),
        "data": r.choices[0].message.parsed.model_dump(),
        "type_safe": True
    }
    
    # JSON mode
    t = time.perf_counter()
    r2 = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "Extract as JSON: product_name, rating(int 1-5), sentiment, pros list, cons list, recommend bool."},
            {"role": "user",   "content": text}
        ]
    )
    results["JSON Mode"] = {
        "latency_ms": round((time.perf_counter() - t) * 1000),
        "data": json.loads(r2.choices[0].message.content),
        "type_safe": False
    }
    
    # Print comparison
    print(f"{'Provider':<20} {'Latency':<12} {'Type Safe':<12} {'Rating':<8} {'Sentiment'}")
    print("-" * 70)
    for provider, res in results.items():
        data = res["data"]
        rating = data.get("rating", "?")
        sentiment = data.get("sentiment", "?")
        print(f"{provider:<20} {res['latency_ms']}ms{'':<8} {'✅' if res['type_safe'] else '❌':<12} {str(rating):<8} {sentiment}")

benchmark_extraction(REVIEW_TEXT)

## 7. LiteLLM — One Call, Any Model

In [ ]:
# LiteLLM normalizes APIs for 100+ providers
# pip install litellm

try:
    import litellm
    litellm.set_verbose = False
    
    schema_hint = '{product_name: str, rating: int 1-5, sentiment: str, recommend: bool}'
    
    models = [
        ("gpt-4o-mini",              "OpenAI"),
        # Add more when you have the keys:
        # ("claude-3-haiku-20240307", "Anthropic"),
        # ("gemini/gemini-1.5-flash", "Gemini"),
    ]
    
    for model_id, provider_name in models:
        try:
            r = litellm.completion(
                model=model_id,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": f"Return JSON: {schema_hint}"},
                    {"role": "user",   "content": REVIEW_TEXT}
                ]
            )
            data = json.loads(r.choices[0].message.content)
            print(f"{provider_name:<12} ({model_id}): rating={data.get('rating')}, recommend={data.get('recommend')}")
        except Exception as e:
            print(f"{provider_name:<12}: Error - {type(e).__name__}")
except ImportError:
    print("📌 Install litellm: pip install litellm")
    print("   Then: litellm.completion(model='gpt-4o-mini', ...)")

## 📌 Summary

| Provider | Method | Schema Strictness |
|---|---|---|
| OpenAI | `.parse(model=..., response_format=MyModel)` | ✅ Token-level |
| Anthropic | Prompt engineering + strip fences | ❌ Best-effort |
| Gemini | `response_mime_type='application/json'` | ⚠️ JSON valid only |
| Any (instructor) | `instructor.from_X(client)` | Provider-dependent |
| Any (LiteLLM) | `litellm.completion(response_format=...)` | Provider-dependent |